# Check that all the necessary simulations were launched

We compare:
- **`parameters.xlsx`** &mdash; the simulations we *wanted* to run.
- **`simulation_statistics_db.csv`** &mdash; the simulations we *actually launched*.

A simulation is identified by the parameters `pai`, `qtrisk`, `seub`, `seum`, `vglow`, `vgmod`.

**Important:** the parameter space is not a full cross-product.
- When `qtrisk == "verifgain"` &rarr; `seub` and `seum` don't matter.
- When `qtrisk` is an `ecart_*` method &rarr; `vglow` and `vgmod` don't matter.

So before comparing we blank out the irrelevant parameters for each method.

In [ ]:
import polars as pl

WORKSPACE = (
    "/home/leyregarrido/01_github_repos/VBR-template/country_pipelines/mali/compile_vbr/workspace"
)

params = pl.read_excel(f"{WORKSPACE}/parameters.xlsx")
stats = pl.read_csv(f"{WORKSPACE}/simulation_statistics_db.csv")

## 1. Put both files in the same shape

Rename the `parameters.xlsx` columns to the parameter names used in the statistics file, and keep only the 6 columns that identify a simulation. String values are stripped of surrounding whitespace.

In [2]:
# Map parameters.xlsx columns -> statistics column names.
wanted = pl.DataFrame(
    {
        "pai": params["paimenet_mode"].str.strip_chars(),
        "qtrisk": params["seuil_max_bas_risk "].str.strip_chars(),
        "seub": params["seuil_max_bas_risk _1"],
        "seum": params["seuil_max_moyen_risk "],
        "vglow": params["verification_gain_low "],
        "vgmod": params["verification_gain_mod "],
    }
)

launched = stats.select(["pai", "qtrisk", "seub", "seum", "vglow", "vgmod"]).with_columns(
    pl.col("pai").str.strip_chars(),
    pl.col("qtrisk").str.strip_chars(),
)

In [3]:
def canonicalize(df: pl.DataFrame) -> pl.DataFrame:
    """Blank out the parameters that don't matter for each risk method, then dedupe."""
    is_vg = pl.col("qtrisk") == "verifgain"
    return df.with_columns(
        # for verifgain, seub/seum are irrelevant
        pl.when(is_vg).then(None).otherwise(pl.col("seub")).alias("seub"),
        pl.when(is_vg).then(None).otherwise(pl.col("seum")).alias("seum"),
        # for the ecart methods, vglow/vgmod are irrelevant
        pl.when(~is_vg).then(None).otherwise(pl.col("vglow")).alias("vglow"),
        pl.when(~is_vg).then(None).otherwise(pl.col("vgmod")).alias("vgmod"),
    ).unique()


wanted_c = canonicalize(wanted)
launched_c = canonicalize(launched)

print(f"Simulations wanted   (unique): {wanted_c.height}")
print(f"Simulations launched (unique): {launched_c.height}")

Simulations wanted   (unique): 40
Simulations launched (unique): 40


## 2. The check

- **Missing** = wanted but not launched &rarr; these still need to be run.
- **Extra** = launched but not wanted &rarr; simulations we ran that weren't asked for (harmless).

`nulls_equal=True` is needed so the blanked-out (null) parameters match each other.

In [4]:
cols = wanted_c.columns

missing = wanted_c.join(launched_c, on=cols, how="anti", nulls_equal=True).sort(cols)
extra = launched_c.join(wanted_c, on=cols, how="anti", nulls_equal=True).sort(cols)

if missing.height == 0:
    print("✅ All necessary simulations have been launched.")
else:
    print(f"❌ {missing.height} simulation(s) are MISSING (wanted but not launched).")

print(f"ℹ️  {extra.height} extra simulation(s) launched that were not in parameters.xlsx.")

✅ All necessary simulations have been launched.
ℹ️  0 extra simulation(s) launched that were not in parameters.xlsx.


In [5]:
# Missing simulations (need to be run)
missing

pai,qtrisk,seub,seum,vglow,vgmod
str,str,f64,f64,i64,i64


In [6]:
# Extra simulations (launched but not requested)
extra

pai,qtrisk,seub,seum,vglow,vgmod
str,str,f64,f64,i64,i64
